In [1]:
!pip install ucimlrepo tensorflow --quiet


In [2]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping


In [3]:
retention_levels = [0.10, 0.25, 0.50, 0.75, 0.90]


In [4]:
def autoencoder_classification_experiment(X, y, retention_levels, epochs=100, batch_size=32):
    results = []

    # Train-test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Standardization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    original_dims = X.shape[1]

    # ------------------
    # Baseline (no DR)
    # ------------------
    baseline_clf = KNeighborsClassifier(n_neighbors=5)
    baseline_clf.fit(X_train_scaled, y_train)
    baseline_acc = accuracy_score(
        y_test, baseline_clf.predict(X_test_scaled)
    ) * 100

    # ------------------
    # Autoencoder experiments
    # ------------------
    for r in retention_levels:
        k = max(1, int(original_dims * r))  # bottleneck dim

        # Build symmetric autoencoder
        input_layer = Input(shape=(original_dims,))
        encoded = Dense(128, activation='relu')(input_layer)
        bottleneck = Dense(k, activation='linear')(encoded)
        decoded = Dense(128, activation='relu')(bottleneck)
        output_layer = Dense(original_dims, activation='linear')(decoded)

        autoencoder = Model(inputs=input_layer, outputs=output_layer)
        autoencoder.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                            loss='mse')

        # Early stopping
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        # Train autoencoder
        autoencoder.fit(X_train_scaled, X_train_scaled,
                        epochs=epochs,
                        batch_size=batch_size,
                        validation_split=0.2,
                        callbacks=[early_stop],
                        verbose=0)

        # Encode train & test
        encoder = Model(inputs=input_layer, outputs=bottleneck)
        X_train_enc = encoder.predict(X_train_scaled)
        X_test_enc = encoder.predict(X_test_scaled)

        # Downstream k-NN classifier
        clf = KNeighborsClassifier(n_neighbors=5)
        clf.fit(X_train_enc, y_train)
        acc = accuracy_score(y_test, clf.predict(X_test_enc)) * 100

        results.append({
            'Retention Level (%)': int(r*100),
            'Bottleneck Dim': k,
            'Accuracy (%)': acc
        })

    return baseline_acc, pd.DataFrame(results)


In [5]:
wine_data = fetch_ucirepo(id=186)
X_wine = wine_data.data.features.values
y_wine = wine_data.data.targets.values.ravel()

baseline_wine, wine_ae = autoencoder_classification_experiment(
    X_wine, y_wine, retention_levels
)

wine_ae['Dataset'] = 'Wine'
wine_ae['Method'] = 'Autoencoder'
wine_ae['Baseline (%)'] = baseline_wine

wine_ae


163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


,Retention Level (%),Bottleneck Dim,Accuracy (%),Dataset,Method,Baseline (%)
0,10,1,44.692308,Wine,Autoencoder,55.846154
1,25,2,48.230769,Wine,Autoencoder,55.846154
2,50,5,56.384615,Wine,Autoencoder,55.846154
3,75,8,55.230769,Wine,Autoencoder,55.846154
4,90,9,56.538462,Wine,Autoencoder,55.846154


In [6]:
breast_data = fetch_ucirepo(id=17)
X_breast = breast_data.data.features.values
y_breast = breast_data.data.targets.values.ravel()

baseline_breast, breast_ae = autoencoder_classification_experiment(
    X_breast, y_breast, retention_levels
)

breast_ae['Dataset'] = 'Breast Cancer'
breast_ae['Method'] = 'Autoencoder'
breast_ae['Baseline (%)'] = baseline_breast

breast_ae


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


,Retention Level (%),Bottleneck Dim,Accuracy (%),Dataset,Method,Baseline (%)
0,10,3,90.350877,Breast Cancer,Autoencoder,95.614035
1,25,7,95.614035,Breast Cancer,Autoencoder,95.614035
2,50,15,92.982456,Breast Cancer,Autoencoder,95.614035
3,75,22,95.614035,Breast Cancer,Autoencoder,95.614035
4,90,27,94.736842,Breast Cancer,Autoencoder,95.614035


In [7]:
(X_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()
X_mnist = X_train.reshape(X_train.shape[0], -1)
y_mnist = y_train

baseline_mnist, mnist_ae = autoencoder_classification_experiment(
    X_mnist, y_mnist, retention_levels
)

mnist_ae['Dataset'] = 'MNIST'
mnist_ae['Method'] = 'Autoencoder'
mnist_ae['Baseline (%)'] = baseline_mnist

mnist_ae


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


,Retention Level (%),Bottleneck Dim,Accuracy (%),Dataset,Method,Baseline (%)
0,10,78,95.083333,MNIST,Autoencoder,94.783333
1,25,196,95.541667,MNIST,Autoencoder,94.783333
2,50,392,95.608333,MNIST,Autoencoder,94.783333
3,75,588,95.525000,MNIST,Autoencoder,94.783333
4,90,705,95.500000,MNIST,Autoencoder,94.783333


In [8]:
ae_table = pd.concat([wine_ae, breast_ae, mnist_ae], ignore_index=True)
ae_table = ae_table[['Dataset', 'Method', 'Baseline (%)',
                     'Retention Level (%)', 'Bottleneck Dim', 'Accuracy (%)']]
ae_table


,Dataset,Method,Baseline (%),Retention Level (%),Bottleneck Dim,Accuracy (%)
0,Wine,Autoencoder,55.846154,10,1,44.692308
1,Wine,Autoencoder,55.846154,25,2,48.230769
2,Wine,Autoencoder,55.846154,50,5,56.384615
3,Wine,Autoencoder,55.846154,75,8,55.230769
4,Wine,Autoencoder,55.846154,90,9,56.538462
5,Breast Cancer,Autoencoder,95.614035,10,3,90.350877
6,Breast Cancer,Autoencoder,95.614035,25,7,95.614035
7,Breast Cancer,Autoencoder,95.614035,50,15,92.982456
8,Breast Cancer,Autoencoder,95.614035,75,22,95.614035
9,Breast Cancer,Autoencoder,95.614035,90,27,94.736842
